# NCKH 05 — Review Diagnostics and DSS Shortlists

This notebook adds review-driven evidence without selecting a new model
or horizon. It reports test-cutoff stability, pre-specified shortlist
quality, and why a recent-zero rule cannot be treated as the target.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

ROOT = Path.cwd()
if not (ROOT / "reports" / "nckh_online_retail_multiclass").exists():
    ROOT = Path("..").resolve()
REPORT = ROOT / "reports" / "nckh_online_retail_multiclass"
SELECTED = REPORT / "selected"
FIG_DIR = REPORT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["CRITICAL", "HIGH_RISK", "MEDIUM_RISK", "LOW_RISK"]
CLASS_COLORS = {
    "CRITICAL": "#B42318",
    "HIGH_RISK": "#E76F00",
    "MEDIUM_RISK": "#176B87",
    "LOW_RISK": "#2A9D8F",
}
MODEL_LABELS = {
    "recent_zero_rule": "Recent-zero rule",
    "recent_quantile_rule": "Recent-sales quantile rule",
    "logistic_regression": "Logistic regression",
    "random_forest": "Random forest",
    "lightgbm": "LightGBM",
}
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

with open(SELECTED / "dataset_profile.json", encoding="utf-8") as handle:
    profile = json.load(handle)
with open(REPORT / "selected_config.json", encoding="utf-8") as handle:
    selected_config = json.load(handle)
with open(SELECTED / "metrics.json", encoding="utf-8") as handle:
    metrics = json.load(handle)

In [ ]:
DIAGNOSTICS = REPORT / "diagnostics"
per_cutoff = pd.read_csv(DIAGNOSTICS / "per_cutoff_metrics.csv")
ranking = pd.read_csv(DIAGNOSTICS / "ranking_metrics.csv")
transition = pd.read_csv(DIAGNOSTICS / "recent_zero_transition.csv")
bootstrap = pd.read_csv(DIAGNOSTICS / "sku_cluster_bootstrap_model_differences.csv")

## Figure 14 — LightGBM stability by test cutoff

In [ ]:
lightgbm = per_cutoff.loc[per_cutoff["model"].eq("lightgbm")].copy()
lightgbm["cutoff_date"] = pd.to_datetime(lightgbm["cutoff_date"])
metrics_to_plot = {
    "macro_f1": "Macro-F1",
    "balanced_accuracy": "Balanced accuracy",
    "high_risk_recall": "High-risk recall",
    "high_risk_pr_auc": "High-risk PR-AUC",
}
fig, ax = plt.subplots(figsize=(9.4, 5.0))
for column, label in metrics_to_plot.items():
    ax.plot(lightgbm["cutoff_date"], lightgbm[column], marker="o", linewidth=2, label=label)
ax.set_ylim(0, 1)
ax.set_xlabel("Test decision cutoff")
ax.set_ylabel("Score")
ax.set_title("LightGBM stability across later test cutoffs")
ax.legend(frameon=False, ncol=2)
ax.grid(alpha=0.2)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIG_DIR / "14_cutoff_stability.png", bbox_inches="tight")
plt.show()

## Figure 15 — Precision@K for DSS shortlists

In [ ]:
shortlist = ranking.loc[ranking["model"].isin(["lightgbm", "random_forest", "logistic_regression", "recent_quantile_rule", "recent_zero_rule"])]
shortlist = shortlist.pivot(index="model_label", columns="k", values="precision_at_k")
shortlist = shortlist.reindex(["LightGBM", "Random forest", "Logistic regression", "Recent-sales quantile rule", "Recent-zero rule"])
fig, ax = plt.subplots(figsize=(10, 5.0))
shortlist.plot(kind="bar", ax=ax, color=["#176B87", "#E76F00"], width=0.75)
ax.set_ylim(0, 1)
ax.set_xlabel("Model")
ax.set_ylabel("Precision of CRITICAL + HIGH_RISK")
ax.set_title("Pre-specified DSS shortlist precision")
ax.legend(title="Shortlist size", frameon=False)
ax.tick_params(axis="x", rotation=18)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "15_precision_at_k.png", bbox_inches="tight")
plt.show()

## Figure 16 — Future class after 28 recent zero-sales days

In [ ]:
transition_plot = transition.loc[transition["cutoff_date"].ne("ALL_TEST_CUTOFFS")].copy()
transition_plot = transition_plot.pivot(index="cutoff_date", columns="risk_class", values="share_within_recent_zero")
transition_plot = transition_plot.reindex(columns=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(9.4, 5.0))
transition_plot.plot(
    kind="bar", stacked=True, ax=ax,
    color=[CLASS_COLORS[name] for name in CLASS_NAMES], width=0.72,
)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_xlabel("Decision cutoff")
ax.set_ylabel("Future-class share among recent-zero SKU snapshots")
ax.set_title("A recent zero-sales period is not deterministically CRITICAL")
ax.legend(title="Future class", frameon=False, ncol=4, loc="upper center")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "16_recent_zero_transition.png", bbox_inches="tight")
plt.show()

## Figure 17 - Paired SKU-cluster bootstrap model differences

In [ ]:
bootstrap_plot = bootstrap.loc[
    bootstrap["reference_model"].isin(
        ["recent_zero_rule", "recent_quantile_rule", "logistic_regression", "random_forest"]
    )
].copy()
display_labels = {
    "recent_zero_rule": "Recent-zero rule",
    "recent_quantile_rule": "Recent-sales quantile rule",
    "logistic_regression": "Logistic regression",
    "random_forest": "Random forest",
}
bootstrap_plot["comparison"] = bootstrap_plot["reference_model"].map(display_labels)
bootstrap_plot["metric_label"] = bootstrap_plot["metric"].map({
    "macro_f1": "Macro-F1",
    "balanced_accuracy": "Balanced accuracy",
    "high_risk_recall": "High-risk recall",
    "high_risk_pr_auc": "High-risk PR-AUC",
})
metric_order = ["Macro-F1", "Balanced accuracy", "High-risk recall", "High-risk PR-AUC"]
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=False)
for ax, metric_label in zip(axes.flat, metric_order):
    data = bootstrap_plot.loc[bootstrap_plot["metric_label"].eq(metric_label)].copy()
    data = data.sort_values("observed_difference")
    y = np.arange(len(data))
    ax.errorbar(
        data["observed_difference"], y,
        xerr=[
            data["observed_difference"] - data["ci_95_lower"],
            data["ci_95_upper"] - data["observed_difference"],
        ],
        fmt="o", color="#176B87", capsize=3,
    )
    ax.axvline(0, color="#B42318", linestyle="--", linewidth=1)
    ax.set_yticks(y, data["comparison"])
    ax.set_title(metric_label)
    ax.set_xlabel("LightGBM - reference model")
    ax.grid(axis="x", alpha=0.2)
fig.suptitle("Paired SKU-cluster bootstrap: 95% confidence intervals", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "17_bootstrap_model_differences.png", bbox_inches="tight")
plt.show()

**Interpretation boundary.** These diagnostics use the already selected
experiment. They do not retune models or select a new horizon. A high
Precision@K measures how concentrated the sales-risk proxy is in a
shortlist; it does not measure inventory savings or prove a business
intervention would be beneficial.